# Analytics Agent - CLI Notebook

A **Text-to-SQL ReAct Agent** that translates natural language questions into SQL queries.

**Supported LLMs:** Groq, Azure OpenAI, Perplexity

# Cell 1: Define your envs

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()


LLM_CONFIG=os.getenv("LLM_CONFIG", "{}")  # Default to empty dict if not set

LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "sk-example-12345")  # Default to example key if not set
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-example-3214123")  # Default to example key if not set
LANGFUSE_BASE_URL = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")  # Default to example URL if not set


GROQ_API_KEY=os.getenv("GROQ_API_KEY", "gsk_example")  # Default to example key if not set

PORT=3050

POSTGRES_DB=os.getenv("POSTGRES_DB", "example")  # Default to example database if not set   
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")  # Default to localhost if not set
POSTGRES_USER = os.getenv("POSTGRES_USER", "example")  # Default to example user if not set
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "example")  # Default to example password if not set
POSTGRES_PORT= int(os.getenv("POSTGRES_PORT", 5432))  # Default to 5432 if not set

## Cell 3: Standard Library Imports

In [16]:
import asyncio
import uuid
import json
import threading
from abc import ABC, abstractmethod
from functools import lru_cache

## Cell 4: Langfuse Setup

In [17]:
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler


langfuse_instance = Langfuse(
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
    environment=os.environ.get("LANGFUSE_ENVIRONMENT", ""),
    base_url=os.environ.get("LANGFUSE_BASE_URL", ""),
)

langfuse_callback = CallbackHandler()

## Cell 5: Tools Abstract Class

In [18]:
import pandas as pd
from langchain_core.tools import StructuredTool


class Tools(ABC):
    @property
    @abstractmethod
    def tools(self) -> list[StructuredTool]:
        ...

    @property
    @abstractmethod
    def tool_get_all_table(self) -> StructuredTool:
        ...

    @property
    @abstractmethod
    def tool_get_table_schema(self) -> StructuredTool:
        ...

    @property
    @abstractmethod
    def tool_execute_query(self) -> StructuredTool:
        ...

    @abstractmethod
    def execute(self, query: str) -> list:
        ...

    @abstractmethod
    def execute_query(self, query: str) -> str:
        ...

    @abstractmethod
    def execute_df(self, query: str) -> pd.DataFrame:
        ...

    @abstractmethod
    def get_all_table(self) -> str:
        ...

    @abstractmethod
    def get_table_schema(self, table_name: str) -> str:
        ...

## Cell 6: PostgresTool Class

In [19]:
import psycopg2

from pydantic import BaseModel, Field


class ExecuteQueryInput(BaseModel):
    query: str = Field(..., description="The SQL query to be executed.")


class GetTableSchemaInput(BaseModel):
    table_name: str = Field(..., description="The name of the table.")


class EmptyInput(BaseModel):
    pass


class PostgresTool(Tools):
    def __init__(self, connection_params: dict):
        self._connection = None
        self.connection_params = connection_params
        self.db_lock = threading.Lock()

    @property
    def connection(self):
        if self._connection is None:
            self._connection = psycopg2.connect(**self.connection_params)
            self._connection.autocommit = True
        return self._connection

    def execute(self, query: str):
        with self.db_lock:
            with self.connection.cursor() as cur:
                cur.execute(query)
                if cur.description:
                    return cur.fetchall()
                return []

    def execute_query(self, query: str) -> str:
        try:
            print("[*] Executing query:", query)
            data = self.execute(query)
            return json.dumps(data[:100], default=str)
        except Exception as e:
            print("[!] Error:", e)
            return str(e)

    def execute_df(self, query: str) -> pd.DataFrame:
        with self.db_lock:
            return pd.read_sql_query(query, self.connection)

    def get_all_table(self, *args, **kwargs) -> str:
        query = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
        """
        tables = self.execute(query)
        return ",".join(t[0] for t in tables)

    def get_table_schema(self, table_name: str, *args, **kwargs) -> str:
        query = f"""
        SELECT
            column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_schema = 'public'
          AND table_name = '{table_name}'
        ORDER BY ordinal_position;
        """
        df = self.execute_df(query)
        if df.empty:
            raise Exception(f"Table '{table_name}' does not exist.")
        return self._df_to_create_statement(df, table_name)

    @staticmethod
    def _df_to_create_statement(df: pd.DataFrame, table_name: str) -> str:
        columns = []
        for _, row in df.iterrows():
            col = f"    {row['column_name']} {row['data_type']}"
            if row["is_nullable"] == "NO":
                col += " NOT NULL"
            columns.append(col)
        return f"CREATE TABLE {table_name} (\n" + ",\n".join(columns) + "\n);"

    @property
    def tool_get_all_table(self):
        return StructuredTool(
            name="get_all_tables",
            description="Retrieve all table names.",
            func=self.get_all_table,
            args_schema=EmptyInput,
        )

    @property
    def tool_get_table_schema(self):
        return StructuredTool(
            name="get_table_create_statement",
            description="Get CREATE TABLE statement.",
            func=self.get_table_schema,
            args_schema=GetTableSchemaInput,
        )

    @property
    def tool_execute_query(self):
        return StructuredTool(
            name="execute_query",
            description="Execute SQL queries.",
            func=self.execute_query,
            args_schema=ExecuteQueryInput,
        )

    @property
    def tools(self):
        return [
            self.tool_execute_query,
            self.tool_get_all_table,
            self.tool_get_table_schema,
        ]

## Cell 7: Initialize db_tool

In [20]:
@lru_cache
def get_postgres_tool() -> Tools:
    return PostgresTool(
        connection_params={
            "host": os.environ.get("POSTGRES_HOST", "localhost"),
            "port": int(os.environ.get("POSTGRES_PORT", "5432")),
            "dbname": os.environ.get("POSTGRES_DB", ""),
            "user": os.environ.get("POSTGRES_USER", ""),
            "password": os.environ.get("POSTGRES_PASSWORD", ""),
        }
    )


db_tool = get_postgres_tool()
print("Database tool initialized.")

Database tool initialized.


## Cell 8: StandaloneTextToSQLAgent Class

In [21]:
from langchain.agents import create_agent
from langchain.chat_models import BaseChatModel
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver


class StandaloneTextToSQLAgent:
    PROMPT_TEMPLATE = """
You are an expert **Text-to-SQL ReAct Agent**. Translate natural language questions into SQL queries.

### Core Directives
1. Use `get_all_tables` and `get_table_create_statement` to retrieve schema when needed.
2. Write efficient SQL queries with `LIMIT 10` for large result sets.
3. Use `execute_query` to run queries.
4. Return only the data results, not the SQL.

### Tool Use (ReAct Pattern: Thought, Action, Observation)
- **get_all_tables**: Get all table names
- **get_table_create_statement**: Get table structure
- **execute_query**: Execute SQL queries

### Constraints
- Do not hallucinate table/column names
- Always execute queries when possible
"""

    def __init__(self, tools: list[StructuredTool], llm: BaseChatModel):
        self.tools = tools
        self.llm = llm
        self._agent = None

    @classmethod
    def from_groq(cls, db_tool: Tools, api_key: str, temperature: float):
        llm = ChatGroq(api_key=api_key, temperature=temperature, model="qwen/qwen3-32b")
        return cls.from_llm(db_tool=db_tool, llm=llm)

    @classmethod
    def from_llm(cls, db_tool: Tools, llm: BaseChatModel):
        if isinstance(db_tool, Tools):
            db_tool = db_tool.tools
        return cls(db_tool, llm)

    @property
    def agent(self):
        if self._agent is None:
            checkpointer = InMemorySaver()
            self._agent = create_agent(
                model=self.llm,
                tools=self.tools,
                system_prompt=self.PROMPT_TEMPLATE,
                checkpointer=checkpointer,
            )
        return self._agent


## Cell 9: Initialize Agent

In [22]:
standalone_text_to_sql_agent = StandaloneTextToSQLAgent.from_groq(
    api_key=os.environ.get("GROQ_API_KEY", ""),
    db_tool=db_tool,
    temperature=0,
)

agent = standalone_text_to_sql_agent.agent

print("Agent initialized with Groq (qwen/qwen3-32b)")

Agent initialized with Groq (qwen/qwen3-32b)


## Cell 10: run_in_thread Function

In [23]:
async def run_in_thread(question: str, thread_id: str):
    _input = {"messages": [{"role": "user", "content": question}]}
    result = await agent.ainvoke(
        _input,
        config={
            "configurable": {"thread_id": thread_id},
            "callbacks": [langfuse_callback],
        }
    )
    return result["messages"][-1].content

## Cell 11: run Function (CLI Loop)

In [24]:
async def run():
    thread_id = str(uuid.uuid4())
    
    while True:
        question = input("Enter your question: ")
        
        if question == "exit":
            break
        
        if question == "new":
            thread_id = str(uuid.uuid4())
            print("Started new conversation thread.")
            continue
        
        answer = await run_in_thread(question, thread_id)
        print("\nAnswer:", answer)

## Cell 12: Run Interactive CLI

In [25]:
# Start interactive CLI session
await run()


Answer: Hello! How can I assist you today?


C:\Users\shtab\AppData\Local\Temp\ipykernel_1788\3437943521.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, self.connection)


[*] Executing query: SELECT product_id, product_name FROM dim_products WHERE category = 'Samsung' AND sub_category = 'Phone' LIMIT 10;
[*] Executing query: SELECT product_id, product_name FROM dim_products WHERE product_name ILIKE '%Samsung%Phone%' LIMIT 10;

Answer: Here are 10 Samsung phone products found by searching product names:

1. TEC-PH-5830 - Samsung Office Telephone, Full Size  
2. TEC-PH-5843 - Samsung Speaker Phone, Cordless  
3. TEC-PH-5829 - Samsung Office Telephone, Cordless  
4. TEC-PH-5840 - Samsung Smart Phone, Full Size  
5. TEC-PH-5841 - Samsung Smart Phone, VoIP  
6. TEC-PH-5846 - Samsung Speaker Phone, with Caller ID  
7. TEC-PH-5832 - Samsung Office Telephone, with Caller ID  
8. TEC-PH-5839 - Samsung Smart Phone, Cordless  
9. TEC-PH-5844 - Samsung Speaker Phone, Full Size  
10. TEC-PH-5831 - Samsung Office Telephone, VoIP  

Would you like to refine this list further (e.g., filter by type like "Smart Phone" or "Speaker Phone") or see more results?
[*] Executin

## Cell 13: Quick Test Query

In [26]:
# # Quick test
# result = asyncio.run(run_in_thread("What tables are available?", str(uuid.uuid4())))
# print("Answer:", result)